# Student 5 — Multi-Task ResNet-18 vs. ResNet-18 + CBAM
**Google Colab notebook**

One image is passed through one shared backbone and produces two predictions:
- **Variety head:** Aseel, Fasli Toto, Gajar, Kupro.
- **Quality head:** Grade-1, Grade-2, Grade-3.

The notebook trains and compares:
1. Baseline multi-task ResNet-18.
2. Multi-task ResNet-18 + CBAM.

It uses the **original dataset only** for the primary experiment. The augmented folder is deliberately not mixed into the split because augmented images may be derived from original images and can cause data leakage.

**Dataset limitation:** In the supplied original dataset, Aseel and Fasli Toto have Grade-1 only. The quality head can learn the three grade labels from the available Gajar and Kupro images, but this dataset cannot demonstrate Grade-2/Grade-3 quality recognition for Aseel or Fasli Toto. The notebook reports this limitation; it does not create missing labels.


## 1. Install dependencies

In [ ]:
!pip -q install torch torchvision scikit-learn pandas matplotlib seaborn pillow tqdm

## 2. Imports, reproducibility, and configuration

In [ ]:
import os, re, random, hashlib, json, time, copy
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Google Drive path confirmed from your folder structure.
# DATA_ROOT must point to the ORIGINAL "Date Fruit" folder,
# not the parent folder and not "Augmented Date Fruit Dataset".
DATA_ROOT = Path(
    "/content/drive/MyDrive/Date Fruit Dataset for Inspection and Grading/"
    "Date Fruit Dataset for Inspection and Grading/Date Fruit"
)

OUTPUT_DIR = Path("/content/student5_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10  # Use the same number of epochs for both models.
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

VARIETIES = ["Aseel", "Fasli Toto", "Gajar", "Kupro"]
GRADES = ["Grade-1", "Grade-2", "Grade-3"]
VARIETY_TO_IDX = {name:i for i,name in enumerate(VARIETIES)}
GRADE_TO_IDX = {name:i for i,name in enumerate(GRADES)}

print("Configured varieties:", VARIETIES)
print("Configured grades:", GRADES)


## 3. Connect Google Drive

Run this cell and approve the access request. The dataset path is already set in the configuration cell above. If your Drive folder differs, edit `DATA_ROOT` there.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("Dataset folder exists:", DATA_ROOT.exists())
print("DATA_ROOT:", DATA_ROOT)
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        "The configured original dataset folder was not found. "
        "Check the Google Drive folder names and update DATA_ROOT."
    )


## 4. Discover images and recover labels from folder names

Your folder structure is `Variety / Size / Grade / image`. We extract Variety and Grade from the relative folder path. Size is recorded for auditing, but it is **not** a prediction target in this two-head experiment.


In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def canonical_variety(path_parts):
    # Match complete folder names, including the two-word label "Fasli Toto".
    for part in path_parts:
        cleaned = part.strip().lower()
        for name in VARIETIES:
            if cleaned == name.lower():
                return name
    return None

def canonical_grade(path_parts):
    for part in path_parts:
        cleaned = part.strip().lower().replace("_", "-").replace(" ", "-")
        for n in [1, 2, 3]:
            if cleaned == f"grade-{n}":
                return f"Grade-{n}"
    return None

def canonical_size(path_parts):
    for part in path_parts:
        for size in ["Large", "Medium", "Small"]:
            if part.strip().lower() == size.lower():
                return size
    return "Unknown"

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"DATA_ROOT not found: {DATA_ROOT}\n"
        "Mount Google Drive and set DATA_ROOT to the original Date Fruit folder."
    )

rows, unmatched = [], []
for p in DATA_ROOT.rglob("*"):
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
        folder_parts = p.relative_to(DATA_ROOT).parts[:-1]
        variety = canonical_variety(folder_parts)
        grade = canonical_grade(folder_parts)
        size = canonical_size(folder_parts)
        if variety is None or grade is None:
            unmatched.append(str(p))
            continue
        rows.append({
            "path": str(p),
            "variety": variety,
            "grade": grade,
            "size": size
        })

df = pd.DataFrame(rows)
print("Recognized original images:", len(df))
print("Images with missing/unrecognized variety or grade:", len(unmatched))
if unmatched[:10]:
    print("Examples of unmatched paths:")
    print("\\n".join(unmatched[:10]))
if df.empty:
    raise ValueError("No labeled images found. Check DATA_ROOT and folder names.")

print("\\nVariety counts:")
display(df["variety"].value_counts().reindex(VARIETIES, fill_value=0))
print("\\nGrade counts:")
display(df["grade"].value_counts().reindex(GRADES, fill_value=0))
print("\\nVariety × Grade:")
display(pd.crosstab(df["variety"], df["grade"]).reindex(
    index=VARIETIES, columns=GRADES, fill_value=0
))
print("\\nVariety × Size × Grade:")
display(df.groupby(["variety", "size", "grade"]).size().reset_index(name="images"))

# The expected total from your latest audit is 3,004 original images.
if len(df) != 3004:
    print(f"WARNING: Found {len(df)} images rather than the previously reported 3004. "
          "Check extensions, duplicate handling, and DATA_ROOT before proceeding.")


## 5. Remove exact duplicate files before splitting
Exact duplicate image bytes are grouped by MD5. If identical files have conflicting labels, stop and inspect the dataset instead of silently assigning one label.

In [ ]:
def md5_file(path, chunk_size=1024*1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

df["md5"] = [md5_file(p) for p in tqdm(df["path"], desc="Hashing images")]
conflicts = df.groupby("md5").agg(
    variety_n=("variety", "nunique"),
    grade_n=("grade", "nunique")
)
bad_hashes = conflicts[(conflicts["variety_n"] > 1) | (conflicts["grade_n"] > 1)]
if len(bad_hashes):
    display(df[df["md5"].isin(bad_hashes.index)].sort_values("md5"))
    raise ValueError("Identical image files have conflicting labels. Resolve these rows before training.")

before = len(df)
df = df.drop_duplicates("md5").reset_index(drop=True)
print(f"Removed {before-len(df)} exact duplicate(s); retained {len(df)} unique images.")

## 6. Remove exact duplicates and create a reproducible 70/15/15 split

We split the **original images only**, stratifying by each observed Variety × Grade combination. Missing combinations (Aseel/Fasli Toto with Grade-2/Grade-3) remain absent; they are not fabricated. Augmentation is applied only to the training transform after splitting.


In [ ]:
def md5_file(path, chunk_size=1024*1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

df["md5"] = [md5_file(p) for p in tqdm(df["path"], desc="Hashing images")]
conflicts = df.groupby("md5").agg(
    variety_n=("variety", "nunique"),
    grade_n=("grade", "nunique")
)
bad_hashes = conflicts[(conflicts["variety_n"] > 1) | (conflicts["grade_n"] > 1)]
if len(bad_hashes):
    display(df[df["md5"].isin(bad_hashes.index)].sort_values("md5"))
    raise ValueError("Identical image files have conflicting labels. Resolve these before training.")

before = len(df)
df = df.drop_duplicates("md5").reset_index(drop=True)
print(f"Removed {before-len(df)} exact duplicate(s); retained {len(df)} unique images.")

df["stratum"] = df["variety"] + "__" + df["grade"]
stratum_counts = df["stratum"].value_counts()
print("\\nObserved Variety × Grade groups:")
display(stratum_counts.rename("images").to_frame())

if stratum_counts.min() < 3:
    raise ValueError(
        "At least one observed Variety × Grade group has fewer than 3 unique images. "
        "A 70/15/15 stratified split may not place that group in all splits."
    )

train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df["stratum"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["stratum"]
)

train_df = train_df.copy(); train_df["split"] = "train"
val_df = val_df.copy(); val_df["split"] = "val"
test_df = test_df.copy(); test_df["split"] = "test"

splits_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
splits_df = splits_df.drop(columns=["stratum"])
splits_csv = OUTPUT_DIR / "dataset_split_manifest.csv"
splits_df.to_csv(splits_csv, index=False)

print("\\nSplit sizes:")
display(splits_df["split"].value_counts())
for split_name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\\n{split_name} Variety × Grade:")
    display(pd.crosstab(part["variety"], part["grade"]).reindex(
        index=VARIETIES, columns=GRADES, fill_value=0
    ))

# Verify exact duplicates do not cross splits.
hash_sets = {s:set(splits_df.loc[splits_df.split==s, "md5"]) for s in ["train","val","test"]}
assert not (hash_sets["train"] & hash_sets["val"])
assert not (hash_sets["train"] & hash_sets["test"])
assert not (hash_sets["val"] & hash_sets["test"])
print("Verified: no exact duplicate hashes cross train/validation/test.")
print("Saved split manifest:", splits_csv)


## 7. Dataset and data loaders
The same normalization and loader settings are used for both models. Training augmentation is mild and only applied to training images.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class DateDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return (
            image,
            torch.tensor(VARIETY_TO_IDX[row["variety"]], dtype=torch.long),
            torch.tensor(GRADE_TO_IDX[row["grade"]], dtype=torch.long),
        )

train_ds = DateDataset(train_df, train_transform)
val_ds = DateDataset(val_df, eval_transform)
test_ds = DateDataset(test_df, eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print(len(train_ds), len(val_ds), len(test_ds))

## 8. Define baseline ResNet-18 and ResNet-18 + CBAM
Both models use an ImageNet-pretrained ResNet-18 backbone and two output heads: 4 variety classes and 3 quality classes. The only architectural difference is the CBAM attention block in the proposed model.

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.channel = ChannelAttention(channels, reduction)
        self.spatial = SpatialAttention(spatial_kernel)
    def forward(self, x):
        x = x * self.channel(x)
        x = x * self.spatial(x)
        return x

class MultiTaskResNet18(nn.Module):
    def __init__(self, use_cbam=False, pretrained=True):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        backbone = models.resnet18(weights=weights)
        self.features = nn.Sequential(*list(backbone.children())[:-2])  # [B, 512, H, W]
        self.attention = CBAM(512) if use_cbam else nn.Identity()
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.variety_head = nn.Linear(512, len(VARIETIES))
        self.quality_head = nn.Linear(512, len(GRADES))
    def forward(self, x):
        x = self.features(x)
        x = self.attention(x)
        x = self.pool(x).flatten(1)
        return self.variety_head(x), self.quality_head(x)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

baseline = MultiTaskResNet18(use_cbam=False, pretrained=True)
proposed = MultiTaskResNet18(use_cbam=True, pretrained=True)
print("Baseline trainable parameters:", f"{count_parameters(baseline):,}")
print("ResNet-18 + CBAM trainable parameters:", f"{count_parameters(proposed):,}")
del baseline, proposed
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 9. Training and validation functions

The model has one shared feature extractor and two task-specific output heads. We use training-set class weights for both losses to reduce the effect of label imbalance. The same weights, split, and training settings are used for both architectures.


In [ ]:
# Compute class weights from TRAINING labels only (never from validation/test).
def make_class_weights(series, class_names):
    counts = series.value_counts().reindex(class_names, fill_value=0).astype(float)
    if (counts == 0).any():
        raise ValueError(f"A class has no training examples: {counts[counts == 0].index.tolist()}")
    weights = len(series) / (len(class_names) * counts)
    return torch.tensor(weights.values, dtype=torch.float32, device=DEVICE)

VARIETY_WEIGHTS = make_class_weights(train_df["variety"], VARIETIES)
QUALITY_WEIGHTS = make_class_weights(train_df["grade"], GRADES)
print("Variety loss weights:", dict(zip(VARIETIES, VARIETY_WEIGHTS.cpu().numpy().round(3))))
print("Quality loss weights:", dict(zip(GRADES, QUALITY_WEIGHTS.cpu().numpy().round(3))))

variety_criterion = nn.CrossEntropyLoss(weight=VARIETY_WEIGHTS)
quality_criterion = nn.CrossEntropyLoss(weight=QUALITY_WEIGHTS)

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    loss_sum, total = 0.0, 0
    all_v_true, all_v_pred, all_q_true, all_q_pred = [], [], [], []

    for images, v_labels, q_labels in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        v_labels = v_labels.to(DEVICE, non_blocking=True)
        q_labels = q_labels.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            v_logits, q_logits = model(images)
            loss_v = variety_criterion(v_logits, v_labels)
            loss_q = quality_criterion(q_logits, q_labels)
            loss = loss_v + loss_q
            if training:
                loss.backward()
                optimizer.step()

        batch_n = images.size(0)
        loss_sum += loss.item() * batch_n
        total += batch_n
        all_v_true.extend(v_labels.detach().cpu().numpy())
        all_v_pred.extend(v_logits.argmax(1).detach().cpu().numpy())
        all_q_true.extend(q_labels.detach().cpu().numpy())
        all_q_pred.extend(q_logits.argmax(1).detach().cpu().numpy())

    return {
        "loss": loss_sum / max(total, 1),
        "variety_acc": accuracy_score(all_v_true, all_v_pred),
        "variety_macro_f1": f1_score(all_v_true, all_v_pred, labels=list(range(len(VARIETIES))), average="macro", zero_division=0),
        "quality_acc": accuracy_score(all_q_true, all_q_pred),
        "quality_macro_f1": f1_score(all_q_true, all_q_pred, labels=list(range(len(GRADES))), average="macro", zero_division=0),
    }

def fit_model(model_name, use_cbam):
    model = MultiTaskResNet18(use_cbam=use_cbam, pretrained=True).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_val_loss = float("inf")
    best_state = None
    history = []

    for epoch in range(1, EPOCHS+1):
        start = time.time()
        train_metrics = run_epoch(model, train_loader, optimizer)
        val_metrics = run_epoch(model, val_loader, optimizer=None)
        row = {"model": model_name, "epoch": epoch}
        row.update({f"train_{k}":v for k,v in train_metrics.items()})
        row.update({f"val_{k}":v for k,v in val_metrics.items()})
        history.append(row)
        print(
            f"{model_name} | epoch {epoch:02d}/{EPOCHS} | "
            f"train loss {train_metrics['loss']:.4f} | val loss {val_metrics['loss']:.4f} | "
            f"val Type Acc {val_metrics['variety_acc']:.4f}, F1 {val_metrics['variety_macro_f1']:.4f} | "
            f"val Quality Acc {val_metrics['quality_acc']:.4f}, F1 {val_metrics['quality_macro_f1']:.4f} | "
            f"{time.time()-start:.1f}s"
        )
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    model_path = OUTPUT_DIR / f"{model_name}_best.pth"
    torch.save({
        "model_name": model_name,
        "model_state_dict": model.state_dict(),
        "varieties": VARIETIES,
        "grades": GRADES,
        "seed": SEED,
        "img_size": IMG_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "best_val_loss": best_val_loss,
        "variety_class_weights": VARIETY_WEIGHTS.cpu(),
        "quality_class_weights": QUALITY_WEIGHTS.cpu(),
    }, model_path)
    history_df = pd.DataFrame(history)
    history_df.to_csv(OUTPUT_DIR / f"{model_name}_history.csv", index=False)
    return model, history_df

baseline_model, baseline_history = fit_model("resnet18_baseline", use_cbam=False)
cbam_model, cbam_history = fit_model("resnet18_cbam", use_cbam=True)


## 10. Plot training curves

In [ ]:
def plot_curves(history_df, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(12,4))
    axes[0].plot(history_df["epoch"], history_df["train_loss"], label="Train")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], label="Validation")
    axes[0].set_title(f"{model_name}: loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend()
    axes[1].plot(history_df["epoch"], history_df["val_variety_macro_f1"], label="Type Macro-F1")
    axes[1].plot(history_df["epoch"], history_df["val_quality_macro_f1"], label="Quality Macro-F1")
    axes[1].set_title(f"{model_name}: validation Macro-F1")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro-F1"); axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_curves(baseline_history, "ResNet-18 baseline")
plot_curves(cbam_history, "ResNet-18 + CBAM")

## 11. Final test evaluation and confusion matrices

The selected checkpoints are evaluated once on the held-out test set. Each task is evaluated separately. Note that the test set has no Aseel/Fasli Toto examples for Grade-2 or Grade-3 because those combinations do not exist in the supplied dataset.


In [ ]:
@torch.no_grad()
def predict_all(model, loader):
    model.eval()
    v_true, v_pred, q_true, q_pred = [], [], [], []
    for images, v_labels, q_labels in tqdm(loader, leave=False):
        images = images.to(DEVICE)
        v_logits, q_logits = model(images)
        v_true.extend(v_labels.numpy())
        q_true.extend(q_labels.numpy())
        v_pred.extend(v_logits.argmax(1).cpu().numpy())
        q_pred.extend(q_logits.argmax(1).cpu().numpy())
    return np.array(v_true), np.array(v_pred), np.array(q_true), np.array(q_pred)

def evaluate_model(model, model_name):
    v_true, v_pred, q_true, q_pred = predict_all(model, test_loader)
    result = {
        "model": model_name,
        "Type Accuracy": accuracy_score(v_true, v_pred),
        "Type Macro-F1": f1_score(v_true, v_pred, labels=list(range(len(VARIETIES))), average="macro", zero_division=0),
        "Quality Accuracy": accuracy_score(q_true, q_pred),
        "Quality Macro-F1": f1_score(q_true, q_pred, labels=list(range(len(GRADES))), average="macro", zero_division=0),
    }

    print("\\n", model_name, "— Variety classification report")
    print(classification_report(
        v_true, v_pred, labels=list(range(len(VARIETIES))),
        target_names=VARIETIES, zero_division=0
    ))
    print(model_name, "— Quality classification report")
    print(classification_report(
        q_true, q_pred, labels=list(range(len(GRADES))),
        target_names=GRADES, zero_division=0
    ))

    print("\\nTest-set support by Variety × Grade (zero means unavailable, not a model failure):")
    display(pd.crosstab(
        test_df["variety"], test_df["grade"]
    ).reindex(index=VARIETIES, columns=GRADES, fill_value=0))

    fig, axes = plt.subplots(1, 2, figsize=(14,5))
    sns.heatmap(
        confusion_matrix(v_true, v_pred, labels=list(range(len(VARIETIES)))),
        annot=True, fmt="d", cmap="Blues",
        xticklabels=VARIETIES, yticklabels=VARIETIES, ax=axes[0]
    )
    axes[0].set_title(f"{model_name}: Variety confusion matrix")
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
    sns.heatmap(
        confusion_matrix(q_true, q_pred, labels=list(range(len(GRADES)))),
        annot=True, fmt="d", cmap="Greens",
        xticklabels=GRADES, yticklabels=GRADES, ax=axes[1]
    )
    axes[1].set_title(f"{model_name}: Quality confusion matrix")
    axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
    plt.tight_layout()
    plt.show()
    return result

baseline_result = evaluate_model(baseline_model, "ResNet-18 baseline")
cbam_result = evaluate_model(cbam_model, "ResNet-18 + CBAM")

results_df = pd.DataFrame([baseline_result, cbam_result])
display(results_df.round(4))
results_df.to_csv(OUTPUT_DIR / "test_metrics_comparison.csv", index=False)
print("Saved test metrics:", OUTPUT_DIR / "test_metrics_comparison.csv")


## 12. Optional robustness evaluation (noise, blur, brightness, reduced resolution)
This optional section tests the same held-out test images under simple synthetic corruptions. These numbers are **additional robustness experiments**, not a replacement for clean test metrics. They should only be reported if your assignment/paper requires them.

In [ ]:
class CorruptedDateDataset(Dataset):
    def __init__(self, frame, corruption="clean"):
        self.frame = frame.reset_index(drop=True)
        self.corruption = corruption
        self.base_transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    def __len__(self): return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.corruption == "brightness_up":
            image = ImageEnhance.Brightness(image).enhance(1.5)
        elif self.corruption == "brightness_down":
            image = ImageEnhance.Brightness(image).enhance(0.6)
        elif self.corruption == "blur":
            image = image.filter(ImageFilter.GaussianBlur(radius=2))
        elif self.corruption == "low_resolution":
            w, h = image.size
            image = image.resize((max(1,w//4), max(1,h//4))).resize((w,h))
        x = self.base_transform(image)
        if self.corruption == "gaussian_noise":
            generator = torch.Generator().manual_seed(SEED + idx)
            noise = torch.randn(x.shape, generator=generator) * 0.08
            x = (x + noise).clamp(-3, 3)
        return x, torch.tensor(VARIETY_TO_IDX[row["variety"]]), torch.tensor(GRADE_TO_IDX[row["grade"]])

@torch.no_grad()
def evaluate_corruption(model, model_name, corruption):
    ds = CorruptedDateDataset(test_df, corruption=corruption)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    v_true, v_pred, q_true, q_pred = [], [], [], []
    model.eval()
    for x, v, q in loader:
        x = x.to(DEVICE)
        vl, ql = model(x)
        v_true.extend(v.numpy()); q_true.extend(q.numpy())
        v_pred.extend(vl.argmax(1).cpu().numpy()); q_pred.extend(ql.argmax(1).cpu().numpy())
    return {
        "model": model_name,
        "corruption": corruption,
        "Type Accuracy": accuracy_score(v_true, v_pred),
        "Type Macro-F1": f1_score(v_true, v_pred, average="macro", zero_division=0),
        "Quality Accuracy": accuracy_score(q_true, q_pred),
        "Quality Macro-F1": f1_score(q_true, q_pred, average="macro", zero_division=0),
    }

# Run if robustness testing is part of the required checkpoint.
RUN_ROBUSTNESS = False
if RUN_ROBUSTNESS:
    robustness_rows = []
    for corruption in ["clean", "brightness_up", "brightness_down", "gaussian_noise", "blur", "low_resolution"]:
        for model, name in [(baseline_model, "ResNet-18 baseline"), (cbam_model, "ResNet-18 + CBAM")]:
            robustness_rows.append(evaluate_corruption(model, name, corruption))
    robustness_df = pd.DataFrame(robustness_rows)
    display(robustness_df.round(4))
    robustness_df.to_csv(OUTPUT_DIR / "robustness_metrics.csv", index=False)

## 13. Save a run configuration and package outputs
Keep the split manifest, model checkpoints, training histories, confusion matrices (you can save the displayed figures if needed), and test metric CSV together.

In [ ]:
config = {
    "experiment": "Multi-task learning: shared ResNet-18 backbone with Variety and Quality heads",
    "data_root": str(DATA_ROOT),
    "dataset_used": "Original Date Fruit only; augmented folder excluded from primary split",
    "seed": SEED,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "split": {"train": 0.70, "validation": 0.15, "test": 0.15},
    "varieties": VARIETIES,
    "grades": GRADES,
    "observed_missing_combinations": [
        "Aseel / Grade-2", "Aseel / Grade-3",
        "Fasli Toto / Grade-2", "Fasli Toto / Grade-3"
    ],
    "loss": "weighted variety cross-entropy + weighted quality cross-entropy; weights computed from training split only",
    "device": str(DEVICE),
}
with open(OUTPUT_DIR / "run_config.json", "w") as f:
    json.dump(config, f, indent=2)
print("Outputs saved in:", OUTPUT_DIR)
print("\\nFiles:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(p.name, f"({p.stat().st_size/1024:.1f} KB)")


## 14. Download outputs from Colab

In [ ]:
from google.colab import files
import shutil
zip_path = shutil.make_archive("/content/student5_outputs", "zip", OUTPUT_DIR)
files.download(zip_path)